# Lab 02-01: Document Extraction

| | |
|---|---|
| **Module** | 02 -- Data Preparation (14% of exam) |
| **Estimated Questions** | ~6 questions |
| **UI Sections** | Data Ingestion, Catalog |
| **Estimated Time** | 45--60 minutes |
| **Difficulty** | Intermediate |

---

## What Is Document Extraction and Why Does It Matter?

Before an LLM can answer questions about your company's data, it needs a
**knowledge base**. But that knowledge does not start as clean text -- it starts
as PDFs, HTML pages, scanned images, Word documents, and spreadsheets sitting
on shared drives and wikis.

**Document extraction** is the first step in every RAG pipeline. It converts
raw files into clean, plain text that can then be chunked, embedded, and stored
in a vector index. If you get extraction wrong, every downstream step suffers:
bad text leads to bad chunks, bad embeddings, and ultimately bad answers.

But even before you extract anything, you must decide **which documents to
include** in your knowledge base. Choosing the wrong sources is the most
common (and most expensive) mistake in RAG system design.

---

## Mental Model

> **Document extraction is like a translator at customs.**
>
> Documents arrive in different formats -- passports, visas, handwritten letters,
> printed forms. The translator's job is to convert every document into a **single
> common language** (clean plain text) so the downstream officers (chunkers,
> embedders) can process them uniformly.
>
> Each document type needs a **different translator** (Python package):
> - Printed text on paper (PDF) -> `pypdf`
> - A website page (HTML) -> `beautifulsoup4`
> - A handwritten or scanned note (image) -> `pytesseract` (OCR)
> - A mix of everything -> `unstructured`
>
> But before you hire translators, you need to decide **which documents to accept
> through customs in the first place**. That is source document identification.

---

## Exam Alert -- Common Traps

| Trap | Why Students Fall For It | The Truth |
|---|---|---|
| Using `pypdf` for scanned images | "It's a PDF, so use pypdf" | If the PDF contains scanned images (no selectable text), `pypdf` returns empty strings. Use `pytesseract`. |
| Using vector search for real-time structured data | "Everything goes in the vector store" | Orders, inventory, and user profiles change constantly. Use a **Feature Serving endpoint** or **SQL query**, not vector search. |
| Skipping source document identification | "Just throw everything in" | Including irrelevant docs increases noise, lowers retrieval precision, and wastes embedding compute. |
| Confusing `beautifulsoup4` with `requests` | "beautifulsoup4 fetches web pages" | `beautifulsoup4` only **parses** HTML. You need `requests` (or similar) to **fetch** the page first. |
| Using `pytesseract` for text-based PDFs | "OCR works on everything" | OCR is slow and error-prone on text that is already machine-readable. Use `pypdf` for text-based PDFs. |

---

## Prerequisites

- Completed **Module 00** (Workspace Orientation, Unity Catalog Basics)
- Familiarity with Python file I/O

### UI Navigation: Uploading Files via Data Ingestion

To upload files to a Volume in Databricks:

1. Click **Catalog** in the left sidebar.
2. Navigate to your catalog -> schema -> Volume (or create one).
3. Click the **Upload to this volume** button (top right).
4. Drag and drop files or browse to select them.
5. Files are now accessible at `/Volumes/<catalog>/<schema>/<volume>/`.

Alternatively, use **Add Data** from the workspace menu -> **Upload files to volume**.

---

## Exam Objectives Covered

- Identify needed source documents that provide necessary knowledge and quality for a given RAG application
- Select the right extraction package for a given document type
- Extract text using `pytesseract`, `pypdf`, and `beautifulsoup4`
- Filter extracted text to remove noise (headers, footers, navigation)

---

## Step 0: Identifying Needed Source Documents

This is the step most teams skip -- and it is the step that determines whether
your RAG application actually works.

Before you extract a single byte of text, you must answer three questions:

1. **What questions will users ask?**
2. **What data answers those questions?**
3. **Where does that data live -- and in what format?**

### The Decision Framework

```
USER QUESTION  -->  What data answers it?  -->  Where does it live?
                                                       |
                              +------------------------+------------------------+
                              |                                                 |
                     UNSTRUCTURED knowledge                          STRUCTURED real-time data
                     (policies, docs, articles)                      (orders, inventory, profiles)
                              |                                                 |
                              v                                                 v
                     Extract -> Chunk -> Embed                        Feature Serving endpoint
                     -> Vector Search Index                           or SQL/API query
```

### The Key Insight

**Not everything belongs in a vector store.**

- **Unstructured knowledge** (policy documents, FAQs, manuals, case law)
  -> Extract, chunk, embed, store in Vector Search. This is the RAG path.

- **Structured real-time data** (order status, inventory counts, user profiles)
  -> Serve from a Feature Serving endpoint or query via SQL/API. This data
  changes too frequently for batch embedding and retrieval by semantic similarity
  makes no sense for exact-match lookups like order IDs.

### Exam-Style Scenario (based on the official exam guide, Q2 pattern)

> *A customer support chatbot answers questions about orders and billing
> correctly, but fails when customers ask about shipping policies. What
> should the team do?*
>
> **Answer:** Add shipping policy documents to the knowledge base
> (extract -> chunk -> embed -> vector search index).
>
> **Why not the other options?**
> - Fine-tuning the LLM? Overkill for adding factual knowledge -- RAG is cheaper and faster.
> - Increasing the model's temperature? That makes output more random, not more accurate.
> - Adding more order data? The bot already handles orders fine -- shipping is the gap.

In [ ]:
# Step 0: Source Document Identification -- Practice Scenarios
#
# For each scenario, we identify:
#   1. What users will ask
#   2. What data source answers those questions
#   3. Whether it belongs in vector search or a feature store / SQL query

SCENARIOS = [
    {
        "app": "FAQ Bot for HR Department",
        "user_asks": "How much PTO do I get? What is the parental leave policy?",
        "data_needed": "Company policy documents + knowledge base articles",
        "data_type": "UNSTRUCTURED",
        "storage": "Vector Search Index",
        "why": (
            "Policy docs are static, unstructured text. Users ask natural-language "
            "questions that require semantic search to find the right policy section."
        ),
    },
    {
        "app": "Order Support Bot",
        "user_asks": "Where is my order #12345? When will it arrive?",
        "data_needed": "Order database / Feature Serving table with order + shipping data",
        "data_type": "STRUCTURED (real-time)",
        "storage": "Feature Serving endpoint or SQL query -- NOT vector search",
        "why": (
            "Order data changes every minute (shipped, delivered, delayed). "
            "You need exact-match lookup by order ID, not semantic similarity. "
            "Embedding order records into a vector store would be stale instantly."
        ),
    },
    {
        "app": "Legal Research Assistant",
        "user_asks": "What precedents exist for patent infringement in software?",
        "data_needed": "Case law documents + statute database",
        "data_type": "UNSTRUCTURED",
        "storage": "Vector Search Index",
        "why": (
            "Case law is long-form unstructured text. Users ask open-ended questions "
            "that require semantic understanding to match relevant cases."
        ),
    },
    {
        "app": "Product Recommender",
        "user_asks": "What laptop should I buy for video editing under $1500?",
        "data_needed": "Product catalog + user preference features",
        "data_type": "STRUCTURED (catalog) + STRUCTURED (preferences)",
        "storage": "Feature Serving endpoint for user prefs + SQL query for catalog filtering",
        "why": (
            "Product specs and prices are structured data that change frequently. "
            "User preferences are structured feature vectors. Neither benefits from "
            "semantic similarity search -- you need exact filters (price < $1500, "
            "category = 'laptop') and feature lookups."
        ),
    },
]


print("=" * 76)
print("SOURCE DOCUMENT IDENTIFICATION -- 4 SCENARIOS")
print("=" * 76)

for i, s in enumerate(SCENARIOS, 1):
    print(f"\n{'~' * 76}")
    print(f"  SCENARIO {i}: {s['app']}")
    print(f"{'~' * 76}")
    print(f"  Users ask     : {s['user_asks']}")
    print(f"  Data needed   : {s['data_needed']}")
    print(f"  Data type     : {s['data_type']}")
    print(f"  Store in      : {s['storage']}")
    print(f"  Why           : {s['why']}")

print(f"\n{'=' * 76}")
print("KEY RULE OF THUMB:")
print("  - Unstructured knowledge (docs, policies, articles) -> Vector Search")
print("  - Structured real-time data (orders, inventory, profiles) -> Feature Serving / SQL")
print("  - If you are doing exact-match lookup (order ID), do NOT use vector search.")
print("=" * 76)

### Step 0 Summary

| Data Type | Example | Where to Store | Why |
|---|---|---|---|
| Unstructured knowledge | Policy docs, FAQs, case law, manuals | Vector Search Index | Semantic search over natural language |
| Structured, static | Product descriptions, article metadata | Could go either way | Depends on query pattern |
| Structured, real-time | Orders, inventory, user profiles | Feature Serving / SQL | Changes too fast for batch embedding; exact-match lookup |

Now that we know **what** to extract, the remaining steps focus on **how** to
extract it.

---

## Step 1: Upload Documents to a Volume

In Databricks, **Volumes** are the Unity Catalog object for storing files --
PDFs, images, CSVs, or any other raw data. Think of a Volume like a shared
network drive that is governed by Unity Catalog permissions.

**Volumes live at:** `/Volumes/<catalog>/<schema>/<volume_name>/`

### Why Volumes?

- **Governed**: Permissions are managed through Unity Catalog (same as tables).
- **Accessible**: Files can be read from notebooks, jobs, and pipelines using
  standard file paths.
- **Integrated**: The Data Ingestion UI lets non-technical users upload files
  via drag-and-drop.

### UI Navigation

1. **Catalog** (left sidebar) -> navigate to your schema.
2. Click **Create** -> **Volume** (if one does not exist).
3. Click into the Volume -> **Upload to this volume** (top right).
4. Drag and drop your source documents.

For this lab, we will simulate by creating sample files programmatically.

In [ ]:
# Step 1: Create sample documents to simulate different file types
#
# In production, these would be real files uploaded to a Volume.
# Here, we create them locally so the lab runs anywhere.

import os
import tempfile

# Create a temporary directory to simulate a Volume path
VOLUME_PATH = tempfile.mkdtemp(prefix="lab_02_01_volume_")
print(f"Simulated Volume path: {VOLUME_PATH}")
print(f"In production, this would be: /Volumes/<catalog>/<schema>/<volume>/\n")

# --- Sample 1: A text file simulating extracted PDF content ---
pdf_text = """ACME Corp Employee Handbook
Page 1 of 25
Last Updated: January 2025
===========================================

Chapter 3: Paid Time Off (PTO)

All full-time employees receive 20 days of PTO per calendar year.
PTO accrues at a rate of 1.67 days per month.
Unused PTO may be carried over up to a maximum of 5 days.

Requesting PTO:
- Submit requests via the HR portal at least 5 business days in advance.
- Requests are approved by your direct manager.
- Requests of 5+ consecutive days require VP-level approval.

===========================================
ACME Corp Confidential -- Do Not Distribute
Page 1 of 25
"""

pdf_path = os.path.join(VOLUME_PATH, "employee_handbook.txt")
with open(pdf_path, "w") as f:
    f.write(pdf_text)


# --- Sample 2: An HTML file simulating a web page ---
html_content = """<!DOCTYPE html>
<html>
<head><title>ACME Corp - Shipping FAQ</title></head>
<body>
  <nav>
    <a href="/">Home</a> | <a href="/products">Products</a> | <a href="/faq">FAQ</a>
  </nav>
  <main>
    <h1>Shipping FAQ</h1>
    <h2>How long does shipping take?</h2>
    <p>Standard shipping takes 5-7 business days. Express shipping takes 2-3 business days.</p>
    <h2>Do you ship internationally?</h2>
    <p>Yes, we ship to over 50 countries. International orders take 10-14 business days.</p>
    <h2>How do I track my order?</h2>
    <p>Log in to your account and visit the Orders page. Click on any order to see tracking details.</p>
  </main>
  <footer>
    <p>Copyright 2025 ACME Corp. All rights reserved.</p>
    <a href="/privacy">Privacy Policy</a> | <a href="/terms">Terms of Service</a>
  </footer>
</body>
</html>
"""

html_path = os.path.join(VOLUME_PATH, "shipping_faq.html")
with open(html_path, "w") as f:
    f.write(html_content)


# --- Sample 3: A plain text file simulating OCR output from a scanned image ---
scanned_text = """INVOICE #20250115-A
Date: January 15, 2025

Bill To:                     Ship To:
Jane Doe                     Jane Doe
123 Main St                  456 Oak Ave
Springfield, IL 62701        Springfield, IL 62704

Item            Qty    Price     Total
Widget A          2   $19.99    $39.98
Widget B          1   $49.99    $49.99
Shipping                        $5.99
-------------------------------------------
TOTAL                          $95.96

Thank you for your business!
"""

scan_path = os.path.join(VOLUME_PATH, "scanned_invoice.txt")
with open(scan_path, "w") as f:
    f.write(scanned_text)


# List all files in our simulated Volume
print("Files in simulated Volume:")
for fname in os.listdir(VOLUME_PATH):
    fpath = os.path.join(VOLUME_PATH, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {fname:30s}  ({size_kb:.1f} KB)")

---

## Step 2: Extract Text from a PDF (`pypdf`)

**When to use `pypdf`:** The PDF contains **selectable text** -- you can highlight
and copy text when viewing it in a PDF reader. Most digitally-created PDFs
(reports, e-books, generated invoices) fall into this category.

**When NOT to use `pypdf`:** The PDF is a scanned image (you cannot select text).
In that case, `pypdf` returns empty strings and you need `pytesseract` (OCR).

### How `pypdf` Works (Simplified)

```
PDF file on disk
    |
    v
PdfReader("file.pdf")         # Open the file
    |
    v
reader.pages                   # Access individual pages
    |
    v
page.extract_text()            # Pull the text from one page
    |
    v
Clean text (remove headers,    # Post-processing
 footers, page numbers)
```

In [ ]:
# Step 2: Extract text from a PDF using pypdf
#
# Since we cannot guarantee pypdf is installed or that a real PDF exists,
# we show the real code pattern AND simulate the output.

# ---------- REAL CODE (run this on Databricks with a real PDF) ----------
# from pypdf import PdfReader
#
# reader = PdfReader("/Volumes/my_catalog/my_schema/my_volume/handbook.pdf")
# full_text = ""
# for page in reader.pages:
#     full_text += page.extract_text() + "\n"
# print(full_text[:500])


# ---------- SIMULATION (works anywhere) ----------
# We will read our sample text file and demonstrate the extraction + cleaning
# pipeline that you would apply to real pypdf output.

with open(pdf_path, "r") as f:
    raw_pdf_text = f.read()

print("=" * 60)
print("RAW EXTRACTED TEXT (before cleaning)")
print("=" * 60)
print(raw_pdf_text)
print(f"\nCharacter count: {len(raw_pdf_text)}")

In [ ]:
# Step 2 (continued): Clean the extracted text
#
# Real PDF extraction produces noise:
#   - Page numbers ("Page 1 of 25")
#   - Headers/footers repeated on every page
#   - Excessive whitespace
#   - Confidentiality notices
#
# Cleaning this text BEFORE chunking improves retrieval quality.

import re


def clean_pdf_text(text: str) -> str:
    """Remove common PDF extraction noise."""
    # Remove page number patterns like "Page X of Y"
    text = re.sub(r"Page \d+ of \d+", "", text)

    # Remove confidentiality footers
    text = re.sub(r".*Confidential.*Do Not Distribute.*", "", text)

    # Remove "Last Updated" lines (metadata, not content)
    text = re.sub(r"Last Updated:.*\n", "", text)

    # Remove separator lines
    text = re.sub(r"={3,}", "", text)

    # Collapse multiple blank lines into one
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


cleaned_pdf_text = clean_pdf_text(raw_pdf_text)

print("=" * 60)
print("CLEANED TEXT (after filtering)")
print("=" * 60)
print(cleaned_pdf_text)
print(f"\nCharacter count: {len(cleaned_pdf_text)} (was {len(raw_pdf_text)})")
print(f"Removed {len(raw_pdf_text) - len(cleaned_pdf_text)} characters of noise")

### Step 2 Takeaway

The `pypdf` extraction pattern is:
1. Open with `PdfReader`
2. Loop through `reader.pages`
3. Call `page.extract_text()` on each page
4. **Clean** the result -- remove headers, footers, page numbers, and noise

Cleaning is not optional. Dirty text degrades chunking and retrieval quality.

---

## Step 3: Extract Text from HTML (`beautifulsoup4`)

**When to use `beautifulsoup4`:** You have HTML files (saved web pages, internal
wiki exports, HTML emails) and need to extract just the meaningful text content,
discarding navigation bars, footers, scripts, and styling.

**Important distinction:** `beautifulsoup4` **parses** HTML. It does not **fetch**
web pages. To download a page, use `requests` first, then parse with BeautifulSoup.

### How BeautifulSoup Works (Simplified)

```
HTML string or file
    |
    v
BeautifulSoup(html, "html.parser")    # Parse the HTML tree
    |
    v
Remove unwanted tags                   # Drop <nav>, <footer>, <script>, <style>
(nav, footer, script, style)
    |
    v
soup.get_text()                        # Extract visible text
    |
    v
Clean whitespace                       # Collapse blank lines
```

In [ ]:
# Step 3: Extract text from HTML using beautifulsoup4

from bs4 import BeautifulSoup

# Read the HTML file
with open(html_path, "r") as f:
    raw_html = f.read()

print("=" * 60)
print("RAW HTML (first 400 characters)")
print("=" * 60)
print(raw_html[:400])
print("...\n")

# Parse the HTML
soup = BeautifulSoup(raw_html, "html.parser")

# Remove unwanted elements: navigation, footer, scripts, styles
for tag in soup.find_all(["nav", "footer", "script", "style"]):
    tag.decompose()  # Completely removes the tag and its contents

# Extract clean text
clean_html_text = soup.get_text(separator="\n", strip=True)

# Collapse multiple blank lines
clean_html_text = re.sub(r"\n{3,}", "\n\n", clean_html_text)

print("=" * 60)
print("CLEANED TEXT (nav/footer/script removed)")
print("=" * 60)
print(clean_html_text)
print(f"\nExtracted {len(clean_html_text)} characters of content")
print(f"Discarded {len(raw_html) - len(clean_html_text)} characters of markup/noise")

### Step 3 Takeaway

The `beautifulsoup4` extraction pattern is:
1. Parse with `BeautifulSoup(html, "html.parser")`
2. Remove noise tags: `soup.find_all(["nav", "footer", "script", "style"])`
3. Extract text: `soup.get_text(separator="\n", strip=True)`
4. Collapse whitespace

The `decompose()` method permanently removes a tag and all its children from
the parse tree. This is how you strip navigation menus, cookie banners, and
footer links that would pollute your chunks.

---

## Step 4: Extract Text from Scanned Images (`pytesseract`)

**When to use `pytesseract`:** You have **scanned documents** -- images (`.jpg`,
`.png`) or PDFs that are essentially photographs of paper. The text is not
machine-readable; it exists only as pixels.

> **This is the #1 tested extraction question on the exam.**
>
> If the question mentions "scanned documents", "images of text",
> "OCR", or "jpg/png files", the answer is **`pytesseract`**.

### How `pytesseract` Works (Simplified)

```
Scanned image (jpg/png)
    |
    v
PIL.Image.open("scan.jpg")          # Load the image
    |
    v
pytesseract.image_to_string(img)    # OCR: convert pixels to text
    |
    v
Raw OCR text (may have errors)      # "rn" misread as "m", etc.
    |
    v
Post-processing / cleaning          # Fix common OCR mistakes
```

### OCR Limitations to Know for the Exam

- **Quality depends on image quality:** Low resolution, skewed, or blurry scans
  produce worse OCR results.
- **Handwriting is unreliable:** `pytesseract` works best on printed text.
- **Tables and columns are tricky:** OCR reads left-to-right, so multi-column
  layouts may produce jumbled text.
- **Slower than other methods:** OCR processes pixels, not encoded text, so it is
  significantly slower than `pypdf` or `beautifulsoup4`.

In [ ]:
# Step 4: Extract text from scanned images using pytesseract
#
# pytesseract requires Tesseract OCR to be installed on the system.
# On Databricks, install via: %pip install pytesseract
# and ensure Tesseract is available (usually pre-installed on ML runtimes).
#
# Since we cannot guarantee Tesseract is installed in all environments,
# we show the REAL code pattern and SIMULATE the output.

# ---------- REAL CODE (run on Databricks with Tesseract installed) ----------
# from PIL import Image
# import pytesseract
#
# # Load the scanned image
# img = Image.open("/Volumes/my_catalog/my_schema/my_volume/scanned_invoice.jpg")
#
# # Run OCR
# ocr_text = pytesseract.image_to_string(img)
#
# print(ocr_text)


# ---------- SIMULATION ----------
# We read our pre-created "scanned" text to demonstrate the post-processing
# pipeline you would apply to real OCR output.

with open(scan_path, "r") as f:
    ocr_text = f.read()

print("=" * 60)
print("SIMULATED OCR OUTPUT (pytesseract.image_to_string)")
print("=" * 60)
print(ocr_text)

# Demonstrate common OCR post-processing
print("\n" + "=" * 60)
print("pytesseract USAGE SUMMARY")
print("=" * 60)
print("""
Essential code pattern:

    from PIL import Image
    import pytesseract

    img = Image.open("scanned_document.jpg")   # or .png
    text = pytesseract.image_to_string(img)     # OCR magic

That's it. Two lines to go from pixels to text.

Exam key facts:
  - pytesseract is for SCANNED IMAGES (jpg, png)
  - It uses Tesseract OCR under the hood
  - It requires PIL/Pillow to load the image
  - It is SLOWER and LESS ACCURATE than pypdf for text-based PDFs
  - Use it ONLY when text is not machine-readable
""")

### Step 4 Takeaway

When the exam says "scanned image" or "OCR", the answer is **`pytesseract`**.
The code is simple -- two lines -- but understanding *when* to use it versus
`pypdf` is what the exam actually tests.

---

## Step 5: The Package Selection Decision Tree

This is the single most important reference for the extraction questions on
the exam. Memorize this decision tree.

In [ ]:
# Step 5: The Package Selection Decision Tree
#
# This is the exam cheat sheet. Memorize this.

decision_tree = """
========================================================================
         DOCUMENT EXTRACTION -- PACKAGE SELECTION DECISION TREE
========================================================================

  What type of file do you have?
  |
  +-- Is it a SCANNED IMAGE? (jpg, png, tiff, or scanned PDF)
  |     |
  |     YES --> pytesseract
  |             (OCR: converts pixels to text)
  |             (Exam keyword: "scanned", "image", "OCR")
  |
  +-- Is it a TEXT-BASED PDF? (you can select/copy text in a viewer)
  |     |
  |     YES --> pypdf
  |             (Reads encoded text directly from the PDF structure)
  |             (Exam keyword: "PDF", "text-based", "selectable")
  |
  +-- Is it HTML? (web page, wiki export, HTML email)
  |     |
  |     YES --> beautifulsoup4
  |             (Parses HTML tree, extracts text, strips tags)
  |             (Exam keyword: "HTML", "web page", "parse")
  |
  +-- Is it a MIX of formats? (or a complex document with tables,
        images, and text all together)
        |
        YES --> unstructured
                (Handles multiple formats with a single API)
                (Exam keyword: "mixed", "complex", "multiple formats")

========================================================================
  QUICK REFERENCE TABLE
========================================================================

  File Type              Package            Key Method
  -------------------    ----------------   ---------------------------
  jpg / png / tiff       pytesseract        image_to_string(img)
  Scanned PDF            pytesseract        image_to_string(img)
  Text-based PDF         pypdf              page.extract_text()
  HTML                   beautifulsoup4     soup.get_text()
  Mixed / complex        unstructured       partition(filename=...)

========================================================================
  THE #1 EXAM TRAP
========================================================================

  Q: "Which package extracts text from scanned documents?"

  WRONG: pypdf        (only works on text-based PDFs)
  WRONG: beautifulsoup4  (only works on HTML)
  WRONG: unstructured (acceptable but NOT the specific answer)

  RIGHT: pytesseract  (OCR -- the exam-correct answer for images)

========================================================================
"""

print(decision_tree)

In [ ]:
# Step 5 (continued): Interactive quiz -- test yourself
#
# Match each scenario to the correct package.

QUIZ = [
    {
        "scenario": "A digitally-generated invoice PDF where you can select and copy text.",
        "answer": "pypdf",
        "explanation": "Text is machine-readable (selectable), so pypdf can extract it directly.",
    },
    {
        "scenario": "A photograph of a whiteboard taken during a meeting.",
        "answer": "pytesseract",
        "explanation": "It is an image (jpg/png). Only OCR (pytesseract) can convert pixels to text.",
    },
    {
        "scenario": "An internal wiki page exported as an HTML file.",
        "answer": "beautifulsoup4",
        "explanation": "HTML content needs to be parsed to extract text and remove tags.",
    },
    {
        "scenario": "A scanned PDF of a signed contract (no selectable text).",
        "answer": "pytesseract",
        "explanation": "Scanned PDF = image. No selectable text means pypdf returns nothing. Use OCR.",
    },
    {
        "scenario": "A folder containing a mix of PDFs, Word docs, and images.",
        "answer": "unstructured",
        "explanation": "Multiple formats in one pipeline -- unstructured handles them all.",
    },
    {
        "scenario": "A .png screenshot of an error message from a user's computer.",
        "answer": "pytesseract",
        "explanation": "A .png file is an image. pytesseract (OCR) is the only option for images.",
    },
]

print("=" * 66)
print("SELF-CHECK QUIZ: Match the Scenario to the Package")
print("=" * 66)

for i, q in enumerate(QUIZ, 1):
    print(f"\n  Q{i}: {q['scenario']}")
    print(f"  --> Answer: {q['answer']}")
    print(f"      Why:    {q['explanation']}")

print("\n" + "=" * 66)
print("If you got all 6 right, you are ready for the extraction questions.")
print("=" * 66)

In [ ]:
# Cleanup: remove temporary files
import shutil

shutil.rmtree(VOLUME_PATH, ignore_errors=True)
print(f"Cleaned up temporary directory: {VOLUME_PATH}")

---

## Exam Tips

| # | Tip | Why It Matters |
|---|---|---|
| 1 | **"Scanned image" = `pytesseract`** -- always. No exceptions on the exam. | This is the most-tested extraction fact. If you see "scanned", "OCR", "jpg", or "png", pick `pytesseract`. |
| 2 | **Identify the data gap first.** If a RAG bot fails on a topic, the fix is to add documents covering that topic -- not fine-tuning or temperature changes. | The exam tests whether you can diagnose missing source data vs. other issues. |
| 3 | **Not everything goes in a vector store.** Structured real-time data (orders, inventory) belongs in Feature Serving or SQL, not vector search. | A common trap answer pairs vector search with order lookups. |
| 4 | **`beautifulsoup4` parses but does not fetch.** You still need `requests` to download the web page first. | The exam may offer `beautifulsoup4` as a web scraping tool -- it is a parser, not a fetcher. |
| 5 | **Always clean extracted text** before chunking. Remove headers, footers, page numbers, and navigation elements. | Dirty text = noisy chunks = poor retrieval = bad answers. The exam expects you to know this. |
| 6 | **`unstructured` is the Swiss-army knife** for mixed-format document collections, but when the exam asks about a specific format (PDF, HTML, image), pick the specific package. | `unstructured` is a valid tool but not the exam-preferred answer for single-format questions. |

---

## Key Takeaways

### Source Document Identification (Step 0)
- Before extracting, decide **which** documents your RAG app needs.
- Framework: What will users ask? -> What data answers that? -> Where does it live?
- Unstructured knowledge -> Vector Search. Structured real-time data -> Feature Serving / SQL.
- If the bot fails on a topic, add documents covering that topic.

### Extraction Package Selection (Steps 2-5)
- `pytesseract` = scanned images (OCR) -- the #1 exam answer for `.jpg` / `.png`.
- `pypdf` = text-based PDFs (selectable text).
- `beautifulsoup4` = HTML parsing (removes tags, extracts text).
- `unstructured` = mixed / multi-format document collections.

### Text Cleaning (Applied in Every Step)
- Remove headers, footers, page numbers, navigation, and boilerplate.
- Collapse whitespace and normalize formatting.
- Clean text BEFORE chunking -- garbage in, garbage out.

---

## Next Lab

**Lab 02-02: Chunking Strategies** takes the clean text produced by this lab
and splits it into chunks suitable for embedding. We will compare fixed-size,
structure-aware, token-based, and hierarchical chunking approaches.